In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_descriptions = pd.read_csv("/content/drive/MyDrive/VRSEC/IV Year/Mini 2/Sahith/Datasets/Edited Dataset/Descriptions.csv")
df_descriptions.drop(index = 2464,inplace = True)
df_descriptions

,DrugID,Descriptions
0,D00001,Polyethylene glycol is a laxative used to trea...
1,D00003,Activated charcoal is a gastric decontaminatio...
2,D00004,Coumarin is an anticoagulant that inhibits the...
3,D00006,Bupropion is a norepinephrine and dopamine reu...
4,D00008,Lindane is an ectoparasiticide and ovicide use...
...,...,...
2459,H01150,Abelmoschus moschatus [Syn. Hibiscus abelmosch...
2460,H01151,Asparagus falcatus is a medicinal plant tradit...
2461,H01152,Barleria prionitis is valued in herbal medicin...
2462,H01154,Solanum xanthocarpum is valued in herbal medic...


# **T5 Model**

In [ ]:
pip install transformers datasets pandas rdkit torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 42.2 MB/s eta 0:00:00


In [ ]:
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

In [ ]:
# Prepare HuggingFace Dataset
dataset = Dataset.from_pandas(df_descriptions[['DrugID', 'Descriptions']])

In [ ]:
# Load T5 tokenizer and model (ChemT5 or vanilla T5)
model_name = "t5-base"  # or use a chem-pretrained version like 'chitholian/chemT5-base'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess(example):
    inputs = tokenizer(example["Descriptions"], truncation=True, padding="max_length", max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["Descriptions"], truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = labels["input_ids"]
    return inputs

In [ ]:
# Prepare training arguments
training_args = TrainingArguments(
    output_dir="./t5-Descriptions-finetuned",
    per_device_train_batch_size=16,
    num_train_epochs=5,
    save_strategy="epoch",
    logging_steps=100,
    eval_strategy="no",
    learning_rate=3e-4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
)

In [ ]:
tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/2464 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
# Data collator for T5 (denoising autoencoding)
# This collator will automatically create decoder_input_ids from the 'labels'
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

/tmp/ipython-input-3119987287.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 238w5a1214 (218w1d7706-vrsec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,0.593800
200,0.003100
300,0.001400
400,0.001700
500,0.000700
600,0.000700
700,0.000400


TrainOutput(global_step=770, training_loss=0.07817923273262266, metrics={'train_runtime': 677.634, 'train_samples_per_second': 18.181, 'train_steps_per_second': 1.136, 'total_flos': 1875590302924800.0, 'train_loss': 0.07817923273262266, 'epoch': 5.0})

In [ ]:
trainer.save_model("./t5-Descriptions-final")
tokenizer.save_pretrained("./t5-Descriptions-final")

('./t5-Descriptions-final/tokenizer_config.json',
 './t5-Descriptions-final/special_tokens_map.json',
 './t5-Descriptions-final/spiece.model',
 './t5-Descriptions-final/added_tokens.json')

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("./t5-Descriptions-final")
model = T5ForConditionalGeneration.from_pretrained("./t5-Descriptions-final")

In [ ]:
from tqdm import tqdm
import torch
import numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load fine-tuned model and tokenizer
tokenizer = T5Tokenizer.from_pretrained("/content/t5-Descriptions-final")
model = T5ForConditionalGeneration.from_pretrained("/content/t5-Descriptions-final")
model.eval()

embeddings = []

# Add tqdm for visibility
with torch.no_grad():
    for Descriptions in tqdm(df_descriptions["Descriptions"], desc="Generating embeddings"):
        inputs = tokenizer(Descriptions, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
        encoder_output = model.encoder(**inputs).last_hidden_state
        mean_embedding = encoder_output.mean(dim=1).squeeze().numpy()  # Mean Pooling
        embeddings.append(mean_embedding)

# Stack into a matrix and join with original DataFrame
matrix = np.vstack(embeddings)
df_result = pd.concat([df_descriptions, pd.DataFrame(matrix)], axis=1)


Generating embeddings: 100%|██████████| 2464/2464 [12:41<00:00,  3.23it/s]


In [ ]:
df_result

,DrugID,Descriptions,0,1,2,3,4,5,6,7,...,758,759,760,761,762,763,764,765,766,767
0,D00001,Polyethylene glycol is a laxative used to trea...,-0.145927,-0.312758,0.130969,0.220534,0.107641,-0.210919,0.116884,-0.050534,...,-0.257689,0.284214,-0.250981,-0.006493,-0.057170,0.094812,0.116533,0.167792,0.139182,0.008030
1,D00003,Activated charcoal is a gastric decontaminatio...,-0.277931,-0.368990,0.014669,0.224999,0.229050,-0.145506,0.136554,0.129338,...,-0.226315,0.218283,-0.227463,0.002175,0.001077,0.035877,0.156483,0.173317,0.145751,0.091981
2,D00004,Coumarin is an anticoagulant that inhibits the...,-0.317302,-0.683028,-0.306363,0.467381,-0.000350,-0.226218,0.056810,-0.060520,...,-0.299282,0.104062,-0.476306,-0.010854,-0.111802,0.189551,0.239954,0.216568,0.011799,-0.123562
3,D00006,Bupropion is a norepinephrine and dopamine reu...,-0.249380,-0.318773,0.094960,0.071642,0.179075,-0.134580,0.146964,-0.014092,...,-0.221909,0.124833,-0.346012,0.071997,-0.078562,0.088697,0.052416,0.227739,-0.047331,0.041158
4,D00008,Lindane is an ectoparasiticide and ovicide use...,-0.053867,-0.352855,0.114435,0.153917,0.117225,-0.057876,0.129429,-0.106695,...,-0.314165,0.155802,-0.226927,-0.013547,-0.019306,0.092635,0.139387,0.091997,0.056408,0.061032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2459,H01150,Abelmoschus moschatus [Syn. Hibiscus abelmosch...,0.166396,-0.111989,0.161319,0.107473,0.075293,-0.017282,0.122791,-0.153552,...,-0.206450,0.123744,-0.146183,-0.155494,-0.065303,-0.047765,0.408979,0.142761,0.085997,0.159785
2460,H01151,Asparagus falcatus is a medicinal plant tradit...,-0.012622,-0.250096,0.249723,0.167884,0.089195,-0.056803,0.083533,-0.085474,...,-0.299773,0.231470,-0.326006,-0.185539,-0.032892,0.088272,0.364158,0.202346,0.134661,0.124522
2461,H01152,Barleria prionitis is valued in herbal medicin...,0.029739,-0.203607,0.086319,0.262230,0.124496,-0.004191,0.127823,0.022505,...,-0.188838,0.079035,-0.093284,-0.061437,-0.040290,0.075986,0.210215,0.093042,-0.029791,0.091464
2462,H01154,Solanum xanthocarpum is valued in herbal medic...,0.093791,-0.180708,0.114484,0.281466,0.141483,0.013864,0.131288,-0.053698,...,-0.239554,0.037699,-0.099342,-0.075540,0.005656,0.089479,0.252977,0.009921,0.010118,0.056794


In [ ]:
# prompt: save the df_result into csv
df_result.to_csv("T5_Descriptions_Embeddings.csv", index=False)